# 1. Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf
import itertools
import sys
import os
import gc
import json
import optuna

# 2. Data & Model Paths

## 1. Folders

In [ ]:
scripts_folder = os.path.abspath(os.path.join('..', 'Scripts'))
data_folder = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
hyper_parameter_folder = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))

## 2. System paths

In [ ]:
sys.path.append(scripts_folder)

# 3. Import Data & Models

## 1. Import Models

In [ ]:
from model import Encoder, Decoder, VAE

## 2. Import Data

In [ ]:
train_data_path = os.path.join(data_folder, 'train_data.npy')

# Import training data
try:
    train_data = np.load(train_data_path).astype(np.float32)
    train_data_tf = tf.constant(train_data, dtype=tf.float32) # Mencegah tensorflow duplikasi data tiap iterasi (memory leak)
    # Mengambil jumlah item 
    num_items = train_data.shape[1]
    print(f"Data latih berhasil dimuat.")
    print(f"   Dimensi data: {train_data.shape} (Pengguna x Film)")
except FileNotFoundError:
    print("Error: File 'train_data.npy' tidak ditemukan. Pastikan Anda sudah menjalankan preprocessing.")

# 4. Hyperparameter Tuning

## 1. VAE (Multinomial)

### 1. Search Space

In [ ]:
# Ruang pencarian hyperparameter VAE untuk Optuna
# Optuna akan menjelajahi ruang ini secara cerdas menggunakan algoritma TPE
vae_search_space = {
    'latent_dim'   : [20, 50, 100, 200],
    'learning_rate': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    'batch_size'   : [64, 128, 256],
    'dropout_rate' : [0.1, 0.2, 0.3, 0.4],
    'hidden_dims'  : [
        [512, 256],
        [512, 256, 128],
        [1024, 512],
        [1024, 512, 256],
    ],
    'beta'         : [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 1.0],
}

# Jumlah trial yang akan dijalankan Optuna
# Optuna memilih kombinasi secara cerdas berdasarkan hasil trial sebelumnya (TPE)
N_TRIALS_VAE = 200

print(f"[INFO] Search space VAE:")
for key, values in vae_search_space.items():
    print(f"       {key:<15}: {values}")
print(f"\n[INFO] Jumlah trial yang akan dijalankan: {N_TRIALS_VAE}")

### 2. Optuna Objective Function

In [ ]:
# Nonaktifkan logging bawaan Optuna agar output lebih bersih
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Pisahkan sebagian train_data sebagai validation set menggunakan rating-level split
# Metode ini konsisten dengan DataPreprocessing.ipynb yang juga menggunakan flat split
VAL_RATIO  = 0.1
np.random.seed(42)

# Ambil koordinat semua rating yang ada
all_rows, all_cols = np.nonzero(train_data)
n_ratings          = len(all_rows)
n_val              = int(n_ratings * VAL_RATIO)

# Acak semua rating lalu ambil 10% pertama sebagai validation
shuffled_idx = np.random.permutation(n_ratings)
val_idx      = shuffled_idx[:n_val]
val_rows     = all_rows[val_idx]
val_cols     = all_cols[val_idx]

# Buat mask validation
val_mask = np.zeros(train_data.shape, dtype=np.float32)
val_mask[val_rows, val_cols] = 1.0

# train_data_tune : data latih (rating validation di-nol-kan)
# val_data_tune   : hanya berisi rating validation
train_data_tune = train_data * (1.0 - val_mask)
val_data_tune   = train_data * val_mask
train_tune_tf   = tf.constant(train_data_tune, dtype=tf.float32)

print(f"[INFO] Total rating training : {int(np.sum(train_data_tune > 0)):,}")
print(f"[INFO] Total rating validasi : {int(np.sum(val_data_tune > 0)):,}")

def vae_objective(trial):
    """
    Fungsi objektif yang dipanggil Optuna untuk setiap trial.
    Optuna menggunakan hasil fungsi ini untuk menentukan kombinasi
    parameter berikutnya yang paling menjanjikan (algoritma TPE).

    Model dilatih pada train_data_tune (90% data) lalu dievaluasi
    reconstruction loss pada val_data_tune (10% held-out).
    Validation split menggunakan rating-level (flat split) yang konsisten
    dengan DataPreprocessing.ipynb. Pendekatan ini mencegah bias terhadap
    beta kecil — model yang overfit akibat KL terlalu lemah akan
    menghasilkan val loss lebih tinggi meski train loss-nya rendah.

    Args:
        trial : objek Optuna yang menyediakan metode suggest_*
                untuk memilih nilai hyperparameter

    Returns:
        val_recon_loss : float, nilai yang diminimalkan Optuna.
    """
    # Optuna memilih nilai untuk setiap hyperparameter dari search space
    latent_dim    = trial.suggest_categorical('latent_dim',    vae_search_space['latent_dim'])
    learning_rate = trial.suggest_categorical('learning_rate', vae_search_space['learning_rate'])
    batch_size    = trial.suggest_categorical('batch_size',    vae_search_space['batch_size'])
    dropout_rate  = trial.suggest_categorical('dropout_rate',  vae_search_space['dropout_rate'])
    beta          = trial.suggest_categorical('beta',          vae_search_space['beta'])

    # hidden_dims dikonversi ke string karena suggest_categorical hanya menerima
    # tipe primitif (str, int, float), lalu dikonversi kembali ke list
    hidden_dims_str = trial.suggest_categorical('hidden_dims', [str(h) for h in vae_search_space['hidden_dims']])
    hidden_dims     = eval(hidden_dims_str)

    # Bersihkan sesi TensorFlow sebelum membuat model baru
    tf.keras.backend.clear_session()
    tf.compat.v1.reset_default_graph()

    # Bangun arsitektur VAE dengan hyperparameter yang dipilih Optuna
    encoder = Encoder(hidden_dims=hidden_dims, latent_dim=latent_dim, dropout_rate=dropout_rate)
    decoder = Decoder(hidden_dims=hidden_dims[::-1], output_dim=num_items)
    vae     = VAE(encoder, decoder, beta=beta)

    dummy_output = vae(train_tune_tf[:1])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    vae.compile(optimizer=optimizer, run_eagerly=True)

    # Latih model pada data training (tanpa rating validation)
    vae.fit(
        train_tune_tf, train_tune_tf,
        epochs     = 30,
        batch_size = batch_size,
        verbose    = 0
    )

    # Evaluasi reconstruction loss pada validation set
    z_mean, _      = vae.encoder.predict(train_tune_tf, verbose=0)
    # Decoder Multinomial menghasilkan logit — terapkan softmax sebelum hitung loss
    logits         = vae.decoder.predict(z_mean, verbose=0)
    reconstruction = tf.nn.softmax(logits, axis=1).numpy()

    # Hitung masked reconstruction loss hanya pada rating validation
    val_mask_tf    = tf.constant(val_mask, dtype=tf.float32)
    val_data_tf    = tf.constant(val_data_tune, dtype=tf.float32)
    squared_diff   = tf.square(val_data_tf - reconstruction)
    masked_se      = squared_diff * val_mask_tf
    val_recon_loss = float(
        tf.reduce_sum(masked_se) / (tf.reduce_sum(val_mask_tf) + 1e-8)
    )

    # Bersihkan memori sebelum kembali ke Optuna
    del encoder, decoder, vae, optimizer, dummy_output, z_mean, _, logits, reconstruction
    gc.collect()
    tf.keras.backend.clear_session()

    return val_recon_loss


### 3. Optuna Study (Resume-able)

In [ ]:
best_vae_path     = os.path.join(hyper_parameter_folder, 'best_vae_weights_mul.weights.h5')
vae_progress_file = os.path.join(hyper_parameter_folder, 'tuning_progress_vae_mul.json')
vae_study_db      = os.path.join(hyper_parameter_folder, 'optuna_vae_mul_study.db')

# Buat atau load kembali study yang sudah ada
# storage SQLite memungkinkan study dilanjutkan jika proses Jupyter terputus
study_vae = optuna.create_study(
    study_name     = 'vae_mul_tuning',
    direction      = 'minimize',
    storage        = f'sqlite:///{vae_study_db}',
    load_if_exists = True    # otomatis resume jika study sudah ada
)

# Hitung berapa trial yang sudah selesai
n_completed = len([t for t in study_vae.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_remaining = max(0, N_TRIALS_VAE - n_completed)

print(f"[INFO] Trial sudah selesai : {n_completed}")
print(f"[INFO] Trial tersisa       : {n_remaining}")

if n_completed > 0:
    print(f"[INFO] Best reconstruction loss sejauh ini: {study_vae.best_value:.6f}")
    print(f"[INFO] Best params sejauh ini             : {study_vae.best_params}")

### 4. Run Tuning

In [ ]:
# Callback untuk mencetak progress setiap trial selesai
def print_trial_callback(study, trial):
    print(f"  Trial {trial.number+1:>3} | Recon Loss: {trial.value:.6f} | Params: {trial.params}")
    if study.best_trial.number == trial.number:
        print(f"         [NEW BEST] Reconstruction Loss turun ke {trial.value:.6f}")

if n_remaining > 0:
    print(f"[INFO] Memulai {n_remaining} trial Optuna (TPE)...\n")
    study_vae.optimize(
        vae_objective,
        n_trials       = n_remaining,
        callbacks      = [print_trial_callback],
        gc_after_trial = True    # bersihkan memori setelah setiap trial
    )
else:
    print("[INFO] Semua trial sudah selesai. Tidak ada yang perlu dijalankan.")

print(f"\n[SELESAI] Best Reconstruction Loss : {study_vae.best_value:.6f}")
print(f"[SELESAI] Best Params               : {study_vae.best_params}")

### 5. Simpan Hasil Terbaik

In [ ]:
# Rekonstruksi hidden_dims dari string kembali ke list
best_params_raw = study_vae.best_params.copy()
best_params_raw['hidden_dims'] = eval(best_params_raw['hidden_dims'])

# Simpan ke format JSON yang sama dengan sebelumnya
# sehingga Training.ipynb, CrossValidation.ipynb, dan Testing.ipynb
# tidak perlu diubah sama sekali
with open(vae_progress_file, 'w') as f:
    json.dump({
        'best_loss'  : study_vae.best_value,
        'best_params': best_params_raw,
    }, f, indent=4)

print(f"[SUCCESS] Parameter terbaik disimpan ke: {vae_progress_file}")
print(f"[INFO]    Isi: {best_params_raw}")